In [1]:
import mlflow
mlflow.__version__

'3.10.0'

In [10]:
import sagemaker_mlflow
sagemaker_mlflow.__version__

'0.4.0'

In [8]:
import boto3
boto3.__version__

'1.43.6'

In [2]:
import yfinance as yf
import pandas as pd
from pathlib import Path

In [7]:
df = yf.download("AAPL", start="2025-01-01", interval='60m', progress=False)
df

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Datetime,,,,,
2025-01-02 14:30:00+00:00,245.739899,249.100006,244.800003,248.929993,13501229
2025-01-02 15:30:00+00:00,244.895004,245.880005,244.160095,245.630005,6830782
2025-01-02 16:30:00+00:00,243.428604,245.250000,243.320007,244.909897,4384552
2025-01-02 17:30:00+00:00,242.686203,243.410004,242.309998,243.410004,6464462
2025-01-02 18:30:00+00:00,242.488403,242.839996,241.820099,242.669998,5547977
...,...,...,...,...,...
2026-06-09 15:30:00+00:00,289.500000,291.980011,289.480011,291.559998,8529475
2026-06-09 16:30:00+00:00,290.459991,291.130005,287.779999,289.500000,8503325


In [ ]:
df = df.reset_index()

df = df.rename(columns={
    df.columns[0]: "date",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Volume": "volume"
})
df

Price,Date,close,high,low,open,volume
Ticker,,AAPL,AAPL,AAPL,AAPL,AAPL
0,2020-01-02,72.400513,72.460776,71.156674,71.409778,135480400
1,2020-01-03,71.696648,72.455966,71.472469,71.629153,146322800
2,2020-01-06,72.267921,72.306491,70.568495,70.819193,118387200
3,2020-01-07,71.928032,72.533072,71.708672,72.277555,108872000
4,2020-01-08,73.085114,73.386431,71.631559,71.631559,132079200
...,...,...,...,...,...,...
1590,2026-05-01,280.140015,287.220001,278.369995,278.859985,79915400
1591,2026-05-04,276.829987,280.630005,274.859985,279.660004,46668400
1592,2026-05-05,284.179993,284.570007,276.500000,276.929993,49311700


In [15]:
df.columns

MultiIndex([(  'Date',     ''),
            ( 'close', 'AAPL'),
            (  'high', 'AAPL'),
            (   'low', 'AAPL'),
            (  'open', 'AAPL'),
            ('volume', 'AAPL')],
           names=['Price', 'Ticker'])

In [4]:
df = pd.read_parquet(Path.cwd().parent / 'data' / 'processed' / 'aapl.parquet')
df

,date,symbol,open,high,low,close,volume,month,day,ma20,rsi14,ATR,return_2,return_5,return_14
0,2020-02-20,AAPL,77.955784,78.443866,76.887794,77.392792,100566000,2,20,77.186467,47.790853,1.809687,76.078667,78.060005,77.070114
1,2020-02-21,AAPL,76.986842,77.429021,75.024840,75.640984,129554000,2,21,77.120935,53.789928,1.713157,77.194977,77.497009,73.608589
2,2020-02-24,AAPL,71.825712,73.497759,69.885457,72.048004,222195200,2,24,76.886843,43.074949,1.930218,76.392792,77.516335,73.403671
3,2020-02-25,AAPL,72.717321,73.099087,69.136424,69.607590,230673600,2,25,76.643543,28.659726,2.024209,74.640984,76.078667,75.860023
4,2020-02-26,AAPL,69.233073,71.975529,69.225825,70.711823,198054800,2,26,76.350115,30.609658,2.118858,71.048004,77.194977,76.486755
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1557,2026-05-01,AAPL,278.859985,287.220001,278.369995,280.140015,79915400,5,1,266.359000,69.310230,6.770711,269.170013,270.059998,258.200012
1558,2026-05-04,AAPL,279.660004,280.630005,274.859985,276.829987,46668400,5,4,267.257500,65.745284,6.844284,270.350006,266.609985,257.829987
1559,2026-05-05,AAPL,276.929993,284.570007,276.500000,284.179993,49311700,5,5,268.791499,65.594806,6.795713,279.140015,269.709991,265.429993
1560,2026-05-06,AAPL,281.920013,288.029999,281.070007,287.510010,58336100,5,6,270.222000,71.071508,6.872140,275.829987,269.170013,262.399994


In [ ]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.2):
        super(LSTMModel, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        
        # 初始化隐藏状态
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # LSTM 前向传播
        out, _ = self.lstm(x, (h0, c0))
        
        # 取最后一个时间步的输出
        out = out[:, -1, :]
        
        # Dropout + 全连接层
        out = self.dropout(out)
        out = self.fc(out)
        
        return out

class EarlyStopping:
    """早停机制"""
    def __init__(self, patience=10, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0
        
        return self.early_stop

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import joblib

def create_sequences(data, target, seq_len=30):
    """创建时间序列样本"""
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(target[i+seq_len])
    return np.array(X), np.array(y)

def compute_technical_indicators(df):
    """计算技术指标"""
    # 创建副本避免警告
    df = df.copy()
    
    # 1. MA 移动平均线
    df['MA5'] = df['close'].rolling(window=5).mean()
    df['MA10'] = df['close'].rolling(window=10).mean()
    df['MA20'] = df['close'].rolling(window=20).mean()
    
    # 2. 价格变化率
    df['price_change'] = df['close'].pct_change()
    df['volume_change'] = df['volume'].pct_change() if 'volume' in df.columns else 0
    
    # 3. RSI (14)
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # 4. ATR (Average True Range)
    high_low = df['high'] - df['low']
    high_close = abs(df['high'] - df['close'].shift())
    low_close = abs(df['low'] - df['close'].shift())
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df['ATR'] = true_range.rolling(window=14).mean()
    
    # 5. MACD
    exp1 = df['close'].ewm(span=12, adjust=False).mean()
    exp2 = df['close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = exp1 - exp2
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_histogram'] = df['MACD'] - df['MACD_signal']
    
    # 6. 波动率
    df['volatility'] = df['close'].rolling(window=20).std()
    
    # 7. 价格位置（当前价格在 N 日内的相对位置）
    df['price_position'] = (df['close'] - df['close'].rolling(20).min()) / (df['close'].rolling(20).max() - df['close'].rolling(20).min())
    
    # 删除 NaN 值
    df = df.dropna()
    
    return df

def prepare_data(df, feature_cols, seq_len=30, val_ratio=0.2):
    """准备训练数据"""
    
    # 1. 计算技术指标
    df = compute_technical_indicators(df)
    
    # 2. 提取特征和标签
    X_raw = df[feature_cols].values
    y_raw = df['close'].shift(-1).fillna(method='ffill').values
    
    # 对齐数据（去掉最后一行因为 label 会 shift）
    X_raw = X_raw[:-1]
    y_raw = y_raw[:-1]
    
    # 3. 时序分割（验证集使用最近 20% 数据）
    split_idx = int(len(X_raw) * (1 - val_ratio))
    
    X_train_raw = X_raw[:split_idx]
    y_train_raw = y_raw[:split_idx]
    X_val_raw = X_raw[split_idx:]
    y_val_raw = y_raw[split_idx:]
    
    # 4. 标准化（只在训练集上 fit）
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_val_scaled = scaler.transform(X_val_raw)
    
    # 5. 创建序列样本
    X_train, y_train = create_sequences(X_train_scaled, y_train_raw, seq_len)
    X_val, y_val = create_sequences(X_val_scaled, y_val_raw, seq_len)
    
    return X_train, y_train, X_val, y_val, scaler

def prepare_data_for_final_training(df, feature_cols, seq_len=30):
    """准备全量数据用于最终训练"""
    
    df = compute_technical_indicators(df)
    
    X_raw = df[feature_cols].values
    y_raw = df['close'].shift(-1).fillna(method='ffill').values
    X_raw = X_raw[:-1]
    y_raw = y_raw[:-1]
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)
    
    X, y = create_sequences(X_scaled, y_raw, seq_len)
    
    return X, y, scaler

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import mlflow
import mlflow.pytorch
import joblib
import argparse
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from model import LSTMModel, EarlyStopping
from preprocess import prepare_data, prepare_data_for_final_training

# 设置随机种子
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

def train_epoch(model, dataloader, optimizer, criterion, device):
    """训练一个 epoch"""
    model.train()
    total_loss = 0
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs.squeeze(), batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def evaluate(model, dataloader, criterion, device):
    """评估模型"""
    model.eval()
    total_loss = 0
    predictions = []
    actuals = []
    
    with torch.no_grad():
        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs.squeeze(), batch_y)
            total_loss += loss.item()
            
            predictions.extend(outputs.cpu().numpy().flatten())
            actuals.extend(batch_y.cpu().numpy().flatten())
    
    # 计算 RMSE
    rmse = np.sqrt(np.mean((np.array(predictions) - np.array(actuals)) ** 2))
    
    return total_loss / len(dataloader), rmse, predictions, actuals

def train_main():
    parser = argparse.ArgumentParser(description='Train LSTM for churn detection')
    parser.add_argument('--data_path', type=str, default='data/stock_data.csv', help='Path to data file')
    parser.add_argument('--epochs', type=int, default=100, help='Number of epochs')
    parser.add_argument('--batch_size', type=int, default=32, help='Batch size')
    parser.add_argument('--hidden_size', type=int, default=128, help='LSTM hidden size')
    parser.add_argument('--num_layers', type=int, default=2, help='Number of LSTM layers')
    parser.add_argument('--learning_rate', type=float, default=0.001, help='Learning rate')
    parser.add_argument('--seq_len', type=int, default=30, help='Sequence length')
    parser.add_argument('--dropout', type=float, default=0.2, help='Dropout rate')
    parser.add_argument('--patience', type=int, default=15, help='Early stopping patience')
    parser.add_argument('--final_train', action='store_true', help='Train on all data')
    parser.add_argument('--experiment_name', type=str, default='churn_lstm', help='MLflow experiment name')
    
    args = parser.parse_args()
    
    # 设置 MLflow experiment
    mlflow.set_experiment(args.experiment_name)
    
    # 特征列
    feature_cols = ['close', 'volume', 'MA5', 'MA10', 'MA20', 'RSI', 'ATR', 
                    'MACD', 'MACD_signal', 'volatility', 'price_position']
    
    # 读取数据
    df = pd.read_csv(args.data_path)
    print(f"Data loaded: {len(df)} rows")
    
    # 判断是否最终全量训练
    if args.final_train:
        # === 阶段二：使用全量数据训练最终模型 ===
        print("=== Stage 2: Final training on all data ===")
        
        # 加载最佳超参数（从之前的实验中找到的）
        best_params = {
            'hidden_size': args.hidden_size,
            'num_layers': args.num_layers,
            'learning_rate': args.learning_rate,
            'dropout': args.dropout,
            'batch_size': args.batch_size,
            'seq_len': args.seq_len
        }
        
        # 准备全量数据
        X_all, y_all, scaler = prepare_data_for_final_training(df, feature_cols, args.seq_len)
        
        # 创建 DataLoader
        dataset = TensorDataset(torch.FloatTensor(X_all), torch.FloatTensor(y_all))
        dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=False)
        
        # 初始化模型
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = LSTMModel(
            input_size=len(feature_cols),
            hidden_size=args.hidden_size,
            num_layers=args.num_layers,
            output_size=1,
            dropout=args.dropout
        ).to(device)
        
        optimizer = optim.Adam(model.parameters(), lr=args.learning_rate)
        criterion = nn.MSELoss()
        
        # 训练
        with mlflow.start_run(run_name="final_model") as run:
            # 记录参数
            mlflow.log_params(best_params)
            mlflow.log_param("train_mode", "final_full_data")
            mlflow.log_param("total_samples", len(X_all))
            
            for epoch in range(args.epochs):
                train_loss = train_epoch(model, dataloader, optimizer, criterion, device)
                mlflow.log_metric("train_loss", train_loss, step=epoch)
                
                if (epoch + 1) % 10 == 0:
                    print(f"Epoch {epoch+1}/{args.epochs}, Loss: {train_loss:.6f}")
            
            # 保存模型和 Scaler
            torch.save(model.state_dict(), "final_model.pth")
            mlflow.log_artifact("final_model.pth", artifact_path="model")
            
            joblib.dump(scaler, "final_scaler.pkl")
            mlflow.log_artifact("final_scaler.pkl", artifact_path="preprocessor")
            
            # 保存为 PyTorch 格式
            mlflow.pytorch.log_model(model, "pytorch_model")
            
            print(f"Final model saved to run: {run.info.run_id}")
    
    else:
        # === 阶段一：时序验证寻找最佳参数 ===
        print("=== Stage 1: Time series validation ===")
        
        # 准备数据（80% 训练，20% 验证 - 验证集是最近的20%）
        X_train, y_train, X_val, y_val, scaler = prepare_data(df, feature_cols, args.seq_len)
        
        print(f"Train samples: {len(X_train)}, Validation samples: {len(X_val)}")
        print(f"Input shape: {X_train.shape[1:]}")

        # 创建 DataLoader
        train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
        val_dataset = TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val))
        train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False)
        
        # 设备
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {device}")
        
        # 开始 MLflow Run
        with mlflow.start_run(run_name=f"lstm_h{args.hidden_size}_l{args.num_layers}") as run:
            # 记录参数
            mlflow.log_params({
                "input_size": len(feature_cols),
                "hidden_size": args.hidden_size,
                "num_layers": args.num_layers,
                "output_size": 1,
                "learning_rate": args.learning_rate,
                "batch_size": args.batch_size,
                "seq_len": args.seq_len,
                "dropout": args.dropout,
                "epochs": args.epochs,
                "patience": args.patience,
                "train_samples": len(X_train),
                "val_samples": len(X_val)
            })
            
            # 记录数据分布信息
            mlflow.log_metrics({
                "train_mean": float(y_train.mean()),
                "train_std": float(y_train.std()),
                "val_mean": float(y_val.mean()),
                "val_std": float(y_val.std())
            })
            
            # 初始化模型
            model = LSTMModel(
                input_size=len(feature_cols),
                hidden_size=args.hidden_size,
                num_layers=args.num_layers,
                output_size=1,
                dropout=args.dropout
            ).to(device)
            
            # 优化器和损失函数
            optimizer = optim.Adam(model.parameters(), lr=args.learning_rate)
            criterion = nn.MSELoss()
            
            # 早停
            early_stopping = EarlyStopping(patience=args.patience)
            
            best_val_loss = float('inf')
            best_predictions = None
            best_actuals = None
            
            # 训练循环
            for epoch in range(args.epochs):
                # 训练
                train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
                
                # 验证
                val_loss, val_rmse, predictions, actuals = evaluate(model, val_loader, criterion, device)
                
                # 记录指标
                mlflow.log_metric("train_loss", train_loss, step=epoch)
                mlflow.log_metric("val_loss", val_loss, step=epoch)
                mlflow.log_metric("val_rmse", val_rmse, step=epoch)
                
                # 保存最佳模型
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_predictions = predictions
                    best_actuals = actuals
                    # 保存模型权重
                    torch.save(model.state_dict(), "best_model.pth")
                    mlflow.log_artifact("best_model.pth", artifact_path="checkpoints")
                
                if (epoch + 1) % 10 == 0:
                    print(f"Epoch {epoch+1}/{args.epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | Val RMSE: {val_rmse:.6f}")
                
                # 早停检查
                early_stopping(val_loss)
                if early_stopping.early_stop:
                    print(f"Early stopping at epoch {epoch+1}")
                    break
            
            # 记录最终指标
            mlflow.log_metric("best_val_loss", best_val_loss)
            
            # 记录预测 vs 实际对比图
            try:
                import matplotlib.pyplot as plt
                plt.figure(figsize=(10, 5))
                plt.plot(best_actuals[:100], label='Actual', alpha=0.7)
                plt.plot(best_predictions[:100], label='Predicted', alpha=0.7)
                plt.title('Predictions vs Actuals (Validation Set)')
                plt.xlabel('Time Step')
                plt.ylabel('Price')
                plt.legend()
                plt.grid(True, alpha=0.3)
                plt.savefig('predictions_plot.png')
                mlflow.log_artifact('predictions_plot.png', artifact_path="plots")
                plt.close()
            except:
                pass
            
            # 保存 Scaler
            joblib.dump(scaler, "scaler.pkl")
            mlflow.log_artifact("scaler.pkl", artifact_path="preprocessor")
            
            # 保存最佳模型
            final_model = LSTMModel(
                input_size=len(feature_cols),
                hidden_size=args.hidden_size,
                num_layers=args.num_layers,
                output_size=1,
                dropout=args.dropout
            )
            final_model.load_state_dict(torch.load("best_model.pth", map_location='cpu'))
            
            # 记录 PyTorch 模型
            mlflow.pytorch.log_model(final_model, "best_pytorch_model")
            
            print(f"\n=== Training Complete ===")
            print(f"Best Validation Loss: {best_val_loss:.6f}")
            print(f"MLflow Run ID: {run.info.run_id}")
            print(f"Model saved at: runs:/{run.info.run_id}/best_pytorch_model")

if __name__ == "__main__":
    train_main()

In [1]:
import mlflow
from mlflow.tracking import MlflowClient
from dotenv import load_dotenv
import os

load_dotenv()  # 加载 .env 文件中的环境变量


# 1. 设置跟踪 URI (重要: 替换为你的 DagsHub 仓库地址)
# 格式通常为: https://dagshub.com/<用户名>/<仓库名>.mlflow
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))

# 2. 找到要删除的实验 ID (注意: 需要用 experiment_id)
experiment_name = "qqq_experiment"  # 替换成你要删除的实验名称
experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment:
    experiment_id = experiment.experiment_id
    print(f"找到实验 '{experiment_name}'，ID 为 {experiment_id}，正在删除...")
    
    # 3. 执行删除操作 (软删除)
    MlflowClient().delete_experiment(experiment_id)
    print("实验已标记为删除。")
else:
    print(f"未找到名为 '{experiment_name}' 的实验。")

2026/05/22 22:45:10 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range
2026/05/22 22:45:11 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


找到实验 'qqq_experiment'，ID 为 0，正在删除...
实验已标记为删除。


In [3]:
from configuration.config import CANDIDATE_PARAMS
from itertools import combinations, product

try_parameters = CANDIDATE_PARAMS

extra_features = try_parameters["features"]["extra_features"]
extra_features_combinations = [[f] for f in extra_features]
for l in range(2, len(extra_features) + 1):
    extra_features_combinations.extend(list(combinations(extra_features, l)))

base_features = try_parameters["features"]["base_features"]

target_columns = try_parameters["targets"]["columns"]

weight_decay_options = try_parameters["targets"]["weight_decay"]
projection_size_options = try_parameters["model"]["projection_size"]
hidden_size_options = try_parameters["model"]["hidden_size"]
num_layers_options = try_parameters["model"]["num_layers"]
seq_len_options = try_parameters["model"]["seq_len"]
lr_options = try_parameters["model"]["lr"]



### try different combinations of parameters
## try different combinations of extra features
for feature_combination, weight_decay, projection_size, hidden_size, num_layers, seq_len, lr in product(
    extra_features_combinations, weight_decay_options, projection_size_options, hidden_size_options, num_layers_options, seq_len_options, lr_options
):
    feature_columns = base_features + list(feature_combination)
    parameters = {
        "model": "lstm_model",
        "input_size": len(feature_columns),
        "feature_columns": feature_columns,
        "target_columns": target_columns,
        "weight_decay": weight_decay,
        "output_size": len(target_columns),
        "projection_size": projection_size,
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "seq_len": seq_len,
        "num_epochs": 100,
        "lr": lr,
        "scaler": None
    }
    print(parameters)

{'model': 'lstm_model', 'input_size': 9, 'feature_columns': ['open', 'high', 'low', 'close', 'volume', 'year', 'month', 'day', 'ma20'], 'target_columns': ['return_2', 'return_5', 'return_14'], 'weight_decay': 0.7, 'output_size': 3, 'projection_size': 32, 'hidden_size': 64, 'num_layers': 2, 'seq_len': 30, 'num_epochs': 100, 'lr': 0.0001, 'scaler': None}
{'model': 'lstm_model', 'input_size': 9, 'feature_columns': ['open', 'high', 'low', 'close', 'volume', 'year', 'month', 'day', 'ma20'], 'target_columns': ['return_2', 'return_5', 'return_14'], 'weight_decay': 0.7, 'output_size': 3, 'projection_size': 32, 'hidden_size': 64, 'num_layers': 2, 'seq_len': 30, 'num_epochs': 100, 'lr': 0.001, 'scaler': None}
{'model': 'lstm_model', 'input_size': 9, 'feature_columns': ['open', 'high', 'low', 'close', 'volume', 'year', 'month', 'day', 'ma20'], 'target_columns': ['return_2', 'return_5', 'return_14'], 'weight_decay': 0.7, 'output_size': 3, 'projection_size': 32, 'hidden_size': 64, 'num_layers': 2, 

In [ ]:
import finnhub
from dotenv import load_dotenv
import os
from datetime import datetime, timedelta

from configuration.config import START_DATE, STOCK

def collect_news(
    stock:str = STOCK,
    start_date: datetime = datetime.strptime(START_DATE, '%Y-%m-%d'),
    end_date: datetime = (datetime.today()- timedelta(days=1))
    ):
    # load Finnhub API key from environment variable
    load_dotenv()
    finnhub_api_key = os.getenv("FINNHUB_API_KEY")
    
    # Initialize Finnhub client
    finnhub_client = finnhub.Client(api_key=finnhub_api_key)
    
    news_list = []
    
    # slide through date range in 1 month intervals to avoid API limits
    for start, end in zip(pd.date_range(start_date, end_date, freq='MS'), pd.date_range(start_date, end_date, freq='ME')):
        # get company news
        company_news = finnhub_client.company_news(stock, _from=start.strftime('%Y-%m-%d'), to=end.strftime('%Y-%m-%d'))
        news_list.extend(company_news)

    # get company news for any remaining date range
    if end < end_date:
        company_news = finnhub_client.company_news(stock, _from=end.strftime('%Y-%m-%d'), to=end_date.strftime('%Y-%m-%d'))
        news_list.extend(company_news)

    company_news_df = pd.DataFrame(news_list)

    # convert unix timestamp to datetime
    company_news_df['datetime'] = pd.to_datetime(company_news_df['datetime'], unit='s')
    
    # sort by datetime
    company_news_df = company_news_df.sort_values(by='datetime').reset_index(drop=True)
    
    #drop na values
    company_news_df = company_news_df.dropna(subset=['headline', 'datetime'])
    
    #drop duplicates
    company_news_df = company_news_df.drop_duplicates(subset=['id']).reset_index(drop=True)
    
    return company_news_df

In [11]:
from urllib import response

from gdeltdoc import GdeltDoc, Filters
import pandas as pd
from dotenv import load_dotenv
import os
from datetime import datetime, timedelta
import requests

from configuration.config import START_DATE, STOCK, SEARCH_KEYWORDS

def collect_news(
    keyword:dict = SEARCH_KEYWORDS[STOCK.upper()],
    start_date: datetime = datetime.strptime(START_DATE, '%Y-%m-%d'),
    end_date: datetime = (datetime.today()- timedelta(days=1))
    ):
    # load Finnhub API key from environment variable
    load_dotenv()

    news_list = []
    
    company_keywords = keyword['company_keywords']  
    company_keyword_query = '"' + '" OR "'.join(company_keywords) + '"'

    industry_keywords = keyword['industry_keywords']
    industry_keyword_query = '"' + '" OR "'.join(industry_keywords) + '"'
    print(f"Collecting news for company keyword: {company_keyword_query} and industry keyword: {industry_keyword_query}")

    # slide through date range in week intervals to avoid API limits
    date_slides = pd.date_range(start_date, end_date, freq='W')
    
    # ensure the first and last date are included in the slides
    if date_slides[0] > start_date:
        date_slides = pd.concat([pd.Series([pd.to_datetime(start_date)]), pd.Series(date_slides)]).reset_index(drop=True)
    if date_slides[-1] < end_date:
        date_slides = pd.concat([pd.Series(date_slides), pd.Series([pd.to_datetime(end_date)])]).reset_index(drop=True)

    for i in range(len(date_slides) - 1):
        start = date_slides[i]
        end = date_slides[i + 1] - timedelta(days=1)  # end date is inclusive, so subtract 1 day
        # get company news
        company_keyword_query = '"' + '" OR "'.join(company_keywords) + '"'

        response = requests.post(
            "https://google.serper.dev/news",
            headers={
                "X-API-KEY": os.getenv("SERPER_API_KEY"),
                "Content-Type": "application/json"
                },
            json={
                "q": company_keyword_query,
                "gl": "my",
                "hl": "en"
            }
        )

        if response.status_code == 200:
            search_results = response.json()
            news_list.extend(search_results.get('news', []))
        else:
            print(f"Search request failed with status code {response.status_code} and message: {response.text}")
    
    news_df = pd.DataFrame(news_list)

    return news_df

collect_news(SEARCH_KEYWORDS[STOCK.upper()], datetime.strptime('2023-01-01', '%Y-%m-%d'), datetime.strptime('2023-02-28', '%Y-%m-%d'))

,title,link,snippet,date,source,imageUrl
0,Inari Amertron Among Most Active Stocks During...,https://bernama.com/en/news.php?id=2562206,"KUALA LUMPUR, May 28 (Bernama) -- Inari Amertr...",3 weeks ago,Bernama,https://encrypted-tbn0.gstatic.com/images?q=tb...
1,Insas trims Inari Amertron holding in RM186m s...,https://theedgemalaysia.com/node/804357,Insas Bhd (KL:INSAS) said its wholly-owned uni...,4 weeks ago,The Edge Malaysia,https://encrypted-tbn0.gstatic.com/images?q=tb...
2,Inari Amertron reports fire incident at Philip...,https://www.thestar.com.my/business/business-n...,KUALA LUMPUR: Inari Amertron Bhd has reported ...,1 month ago,The Star,https://encrypted-tbn0.gstatic.com/images?q=tb...
3,Inari Amertron rallies despite Q3 profit plung...,https://www.freemalaysiatoday.com/category/hig...,Inari Amertron Bhd shares jumped nearly 13% to...,3 weeks ago,Free Malaysia Today,https://encrypted-tbn0.gstatic.com/images?q=tb...
4,Inari Amertron faces short-term disruptions af...,https://www.nst.com.my/business/corporate/2026...,KUALA LUMPUR: Inari Amertron Bhd is likely to ...,1 month ago,NST Online,https://encrypted-tbn0.gstatic.com/images?q=tb...
...,...,...,...,...,...,...
85,Inari Amertron appoints Phang Ah Tong as chairman,https://www.klsescreener.com/v2/news/view/1735...,Inari Amertron Bhd (KL:INARI) has appointed it...,1 week ago,KLSE Screener,https://encrypted-tbn0.gstatic.com/images?q=tb...
86,"Insider Moves: Inari Amertron Bhd, Hong Seng C...",https://theedgemalaysia.com/node/805833,Notable filings During the week of May 18 to 2...,1 week ago,The Edge Malaysia,https://encrypted-tbn0.gstatic.com/images?q=tb...
87,Inari Amertron reports fire incident at Philip...,https://www.thestar.com.my/business/business-n...,KUALA LUMPUR: Inari Amertron Bhd has reported ...,1 month ago,The Star,https://encrypted-tbn0.gstatic.com/images?q=tb...
88,Stock Today: Inari Amertron Falls 0.9% Despite...,https://www.businesstoday.com.my/2026/06/10/st...,Inari Amertron shares slipped 0.88% despite a ...,6 days ago,BusinessToday Malaysia,https://encrypted-tbn0.gstatic.com/images?q=tb...


In [10]:
import pandas as pd
from datetime import datetime, timedelta
from configuration.config import GDELT_KEYWORD
import requests
from dotenv import load_dotenv
import os
import time

start = '20230101000000'
end = '20230630235959'

load_dotenv()
gdelt_api_key = os.getenv("GDELT_API_KEY")

url = "https://gdeltcloud.com/api/v2/doc"
headers = {
    "Authorization": f"Bearer {gdelt_api_key}"
}
params = {
    "query": GDELT_KEYWORD['company_keyword'],
    "mode": "artlist",
    "format": "json",
    "maxrecords": 250,
    "startdatetime": start,
    "enddatetime": end,
}

response = requests.get(url, headers=headers, params=params)
if response.status_code == 200:
    print(response.json())
elif response.status_code == 429:
    retry_after = response.headers.get("Retry-After")
    if retry_after:
        print(f"Recieved retry after header: {retry_after} seconds. Waiting before retrying...")
        # time.sleep(int(retry_after))
    else:
        print("Rate limit exceeded but no retry after header found. Waiting for 60 seconds before retrying...")
        # time.sleep(60)
else:
    print(f"Error: Received status code {response.status_code} with message: {response.text}")


Error: Received status code 404 with message: <!DOCTYPE html><!--8gK6PU9GbY2_fxsbl_26u--><html lang="en"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="preload" as="image" imageSrcSet="/_next/image?url=%2Fgdelt_cloud_logo_transparent.png&amp;w=48&amp;q=75 1x, /_next/image?url=%2Fgdelt_cloud_logo_transparent.png&amp;w=96&amp;q=75 2x"/><link rel="stylesheet" href="/_next/static/chunks/1157e561d46911fb.css?dpl=dpl_2EM7GtQyYfwqkSGvN4h42gYoJA89" data-precedence="next"/><link rel="stylesheet" href="/_next/static/chunks/d489d2b5519f165f.css?dpl=dpl_2EM7GtQyYfwqkSGvN4h42gYoJA89" data-precedence="next"/><link rel="preload" as="script" fetchPriority="low" href="/_next/static/chunks/53b2816be16629da.js?dpl=dpl_2EM7GtQyYfwqkSGvN4h42gYoJA89"/><script src="/_next/static/chunks/54eafa0d405d8daa.js?dpl=dpl_2EM7GtQyYfwqkSGvN4h42gYoJA89" async=""></script><script src="/_next/static/chunks/8cdf6f0a90dda5c4.js?dpl=dpl_2EM7GtQyYfwqkSGvN4h42gYoJA

In [29]:
import os
import requests

headers = {"Authorization": f"Bearer {os.environ['GDELT_API_KEY']}"}
resp = requests.get(
    "https://gdeltcloud.com/api/v2/stories",
    headers=headers,
    params = {
    "query": "Apple OR AAPL",
    "date_start": "2025-07-01",
    "date_end": "2025-07-30",
    "limit":250
},
)
print(resp.status_code)
if resp.status_code == 200:
    json = resp.json()
    data = json.get("data", None)
    print(data)
elif resp.status_code == 429:
    print(f"Rate limit exceeded.Returned wait time {resp.headers.get('Retry-After')}.")
elif resp.status_code == 400:
    print(f"Bad request. Response: {resp.text}")

200
[]


[{'title': 'Inari Amertron Among Most Active Stocks During Early Trade Amid Stronger Outlook', 'link': 'https://bernama.com/en/news.php?id=2562206', 'snippet': 'KUALA LUMPUR, May 28 (Bernama) -- Inari Amertron Berhad emerged as one of the most actively traded stocks in early trading today,...', 'date': '3 weeks ago', 'source': 'Bernama', 'imageUrl': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRm-DJFpawCAay8_AA2xhsoRTMdryhjMWknRtI6w1VNcsSEvhRapggc6vQ&usqp=CAI&s'}, {'title': 'Capital A, Tanco, Duopharma, Inari Amertron, Dayang, Kee Ming, Sime Darby Property', 'link': 'https://theedgemalaysia.com/node/806431', 'snippet': 'Here is a brief recap of some business news and corporate announcements that made the headlines on Tuesday.', 'date': '6 days ago', 'source': 'The Edge Malaysia', 'imageUrl': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQO75MurChzrJcHLo3dfPNHLXM1DFgZIk5ZuvYjDMDcFEGGa6ibOoVfj-o&usqp=CAI&s'}, {'title': 'Inari on the rise with RF demand', 'link': 'https://www.thestar.com.my/business/business-news/2026/05/29/inari-on-the-rise-with-rf-demand', 'snippet': "Inari Amertron Bhd's earnings prospects are expected to strengthen over the next two financial years, underpinned by a recovery in radio...", 'date': '2 weeks ago', 'source': 'The Star', 'imageUrl': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQSLkPk6UNi-FPwfngNWTDERRw1kCqNdjdknmHd_SPBz6hEeEU1sbN9wKg&usqp=CAI&s'}, {'title': 'Stock Picks: Inari Amertron And SP Setia', 'link': 'https://www.businesstoday.com.my/2026/06/10/stock-picks-inari-amertron-and-sp-setia/', 'snippet': 'RHB Research has identified Inari Amertron and SP Setia as its latest stock picks, citing bullish technical indicators and potential upside...', 'date': '6 days ago', 'source': 'BusinessToday Malaysia', 'imageUrl': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSmQQXTbN7o9ihfdv2hOpBaEb9x4Cg0fxe4MVa3tCA&usqp=CAI&s'}, {'title': 'Inari Amertron rallies despite Q3 profit plunging 50%', 'link': 'https://www.freemalaysiatoday.com/category/highlight/2026/05/26/inari-amertron-rallies-despite-q3-profit-plunging-50', 'snippet': 'Inari Amertron Bhd shares jumped nearly 13% today despite posting a disappointing set of results for its third quarter ended March 31 (Q3...', 'date': '3 weeks ago', 'source': 'Free Malaysia Today', 'imageUrl': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTNsdP073WxkNlneUuCxgJ7ypC2q4FJ4xYJirvlG5sB6c9JNUbzFd9jEn8&usqp=CAI&s'}, {'title': 'Inari Amertron Appoints Phang Ah Tong As Chairman', 'link': 'https://theexchangeasia.com/inari-amertron-appoints-phang-ah-tong-as-chairman/', 'snippet': "Inari Amertron Bhd has appointed its independent director Datuk Phang Ah Tong as the group's new chairman, effective Tuesday, as part of a...", 'date': '5 days ago', 'source': 'The Exchange Asia', 'imageUrl': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRYzg0-TF_Lfjq4UkePTP4an3PqvMzJuyptHL4hBUecdWaRsLGT7EYc3RE&usqp=CAI&s'}, {'title': 'Insas trims Inari Amertron holding in RM186m share sale', 'link': 'https://theedgemalaysia.com/node/804357', 'snippet': 'Insas Bhd (KL:INSAS) said its wholly-owned units Insas Plaza Sdn Bhd and Insas Technology Bhd have sold 100 million shares in Inari Amertron...', 'date': '4 weeks ago', 'source': 'The Edge Malaysia', 'imageUrl': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcR8nS5kkuztzvjOnBbnZDVAIuiOcSM5ZWKUilxT5EhUbrtXyubkBFnz8Qk&usqp=CAI&s'}, {'title': 'Inari Amertron faces short-term disruptions after fire at Philippines plant', 'link': 'https://www.nst.com.my/business/corporate/2026/05/1438364/inari-amertron-faces-short-term-disruptions-after-fire', 'snippet': 'KUALA LUMPUR: Inari Amertron Bhd is likely to see a temporary operational disruptions following a fire incident that took place on May 10 at...', 'date': '1 month ago', 'source': 'NST Online', 'imageUrl': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTHltg-jYAp8t7lDc_KtChlulwgJ8qy8AWwIKiLx7ZW1Pz6RW74W2-ofR4&usqp=CAI&s'}, {'title': 'Stock Today: Inari Amertron Falls 0.9% Despite Positive Technical Outlook', 'link': 'https://www.businesstoday.com.my/2026/06/10/stock-today-inari-amertron-falls-0-9-despite-positive-technical-outlook/', 'snippet': 'Inari Amertron shares slipped 0.88% despite a new chairman appointment and a bullish technical call from RHB Research, which sees potential...', 'date': '5 days ago', 'source': 'BusinessToday Malaysia', 'imageUrl': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcT2TINaTfImyjhDMA30biu9tAdb4_ExLiBUqkem3Um0UPcWLykgRPlfgLc&usqp=CAI&s'}, {'title': 'Inari Amertron reports fire incident at Philippines plant', 'link': 'https://www.thestar.com.my/business/business-news/2026/05/12/inari-amertron-reports-fire-incident-at-philippines-plant', 'snippet': 'KUALA LUMPUR: Inari Amertron Bhd has reported a fire that broke out at the manufacturing facility of Amertron Incorporated in Philippines...', 'date': '1 month ago', 'source': 'The Star', 'imageUrl': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRRGl0b0bgoeH5JOCyBQiJT8coYYy71TY2yZJHQHY_4hsjd0wVVh4J2B9U&usqp=CAI&s'}]

In [27]:
from gdeltdoc import GdeltDoc, Filters
filters = Filters(
    keyword='Inari Amertron',
    start_date='2025-07-01',
    end_date='2025-07-30'
)
gd = GdeltDoc()
gdelt_docs = gd.article_search(filters=filters)
gdelt_docs

RateLimitError: 